# Seizure-File LORO — Fixed Operating Point (no tuning)

Leave-one-record-out over the **seizure-containing files only**. Each fold holds out one seizure file, trains a fixed SVM on all the other seizure files (in full), and scores the held-out file in full at a fixed operating point. No hyperparameter search, no threshold/k calibration, no non-seizure loop.

## Imports

In [1]:
import os
from pathlib import Path

# Single-thread BLAS so the outer joblib parallelism over folds is clean.
for env_var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[env_var] = "1"

import numpy as np
import pandas as pd
import h5py
from joblib import Parallel, delayed

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, average_precision_score

from timescoring import scoring
from timescoring.annotations import Annotation

## CONFIG

In [2]:
# --- Patients to run ---
# Default to chb01..chb24; the PATIENT_SHARD env var can subset the list.
ALL_PATIENTS = [f"chb{n:02d}" for n in range(1, 25)]
shard = os.environ.get("PATIENT_SHARD", "").strip()
if shard:
    PATIENTS = [p.strip() for p in shard.split(",") if p.strip()]
else:
    PATIENTS = ALL_PATIENTS

# --- Paths ---
SEIZURE_DIR     = Path("Features/Seizure")
ANNOTATIONS_CSV = "seizure_annotations.csv"
FEATURE_KEY     = "all_feats"      # dataset name inside the .mat files
LABEL_KEY       = "all_labels"
N_FEATURES      = 1530             # used only to detect transposed arrays

# --- Fixed model config (NOT tuned) ---
# Pipeline: StandardScaler -> mRMR(k_feat) -> RBF-SVC with sqrt-damped class
# weights. Every fold uses exactly these values; nothing is searched.
CLASS_WEIGHT_POWER = 0.5
cfg = {"C": 1, "gamma": 0.001, "k_feat": 100}

# --- Windowing (must match feature extraction) ---
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1

# --- Fixed operating point (a-priori; no calibration) ---
# threshold = 0.0 -> SVM native boundary; k = 3 -> sustained-detection persistence.
FIXED_THRESHOLD = 0.0
FIXED_K         = 3
K_SWEEP         = [2, 3, 4]        # event metrics also reported at each k (reporting only)

# --- SzCORE event-scoring parameters (Dan et al., 2024) ---
SZCORE_PARAMS = scoring.EventScoring.Parameters(
    toleranceStart=30, toleranceEnd=60, minOverlap=0,
    maxEventDuration=5 * 60, minDurationBetweenEvents=90,
)

# --- Parallelism over LORO folds ---
OUTER_N_JOBS = 4

# --- Outputs ---
SUMMARY_CSV = "RESULTS_SUMMARY.csv"   # combined summary across patients

SEIZ = pd.read_csv(ANNOTATIONS_CSV)


# FCQ-mRMR feature selection (aligned to the reference `mrmr` package).
# Redundancy-aware selection: EEG feature banks (band powers x channels x
# derived ratios) are heavily collinear, so pure-relevance top-k (MI) wastes
# the budget on correlated copies of the same signal. mRMR maximises relevance
# while penalising redundancy -> compact, diverse set (better in p >> n).
#   relevance  = ANOVA F-statistic (f_classif)          -- fast, scale-invariant
#   redundancy = mean |Pearson corr| to already-picked features
#   greedy     = argmax  relevance / redundancy  (FCQ quotient variant)
# Deterministic (no RNG) -> also removes a selection-variance source vs MI.
def mrmr_rank(X, y, k, floor=1e-3):
    """Feature indices ordered by greedy FCQ-mRMR (len <= k). Relevance = ANOVA
    F-statistic (f_classif); redundancy = mean |Pearson corr| to already-selected
    features; score = relevance / redundancy (quotient / FCQ form). ALIGNED to the
    reference `mrmr` package (Mazzanti, the FCQ default): candidate pool restricted
    to relevance>0, each pairwise redundancy floored at `floor` (=0.001), and a
    feature whose mean redundancy to the selected set is 1.0 is rejected (score 0).
    Greedy-forward, so order[:m] is the m-feature subset for any m<=k -> compute
    once at max(k) and slice. Matches the package bit-for-bit on well-conditioned
    data; a 1-2 feature residual can appear only in float-tie / exact-duplicate
    edge cases (immaterial)."""
    X = np.asarray(X, float); y = np.asarray(y)
    n, p = X.shape
    with np.errstate(all="ignore"):
        F, _ = f_classif(X, y)
    F = np.nan_to_num(np.asarray(F, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    F = np.maximum(F, 0.0)
    pool = F > 0.0                                      # package: keep only relevance>0
    k = int(min(k, int(pool.sum())))
    if k <= 0:
        return np.array([], dtype=int)
    # z-score (guard zero-variance) so col dot-products == Pearson correlations
    sd = X.std(0); sd = np.where(sd < 1e-12, 1.0, sd)
    Z = (X - X.mean(0)) / sd
    remaining = pool.copy()
    first = int(np.argmax(np.where(pool, F, -np.inf)))  # step 1: most relevant
    selected = [first]; remaining[first] = False
    accum = np.zeros(p)                                 # sum of FLOORED |corr| to picks
    for _ in range(1, k):
        corr = np.abs((Z.T @ Z[:, selected[-1]]) / n)   # |corr| vs last pick
        corr = np.clip(np.nan_to_num(corr, nan=floor), floor, None)   # per-pair floor
        accum += corr
        den = accum / len(selected)                     # mean |corr| to selected
        den = np.where(np.isclose(den, 1.0), np.inf, den)   # reject perfect-redundant
        score = np.where(remaining, F / den, -np.inf)
        nxt = int(np.argmax(score))
        if not np.isfinite(score[nxt]):
            break
        selected.append(nxt); remaining[nxt] = False
    return np.array(selected, dtype=int)


class MRMRSelector(BaseEstimator, TransformerMixin):
    """Top-k FCQ-mRMR selector. Ranking is fit on TRAINING data only, so it is
    leakage-free inside a Pipeline / CV split (drop-in for SelectKBest)."""
    def __init__(self, k=20):
        self.k = k
    def fit(self, X, y=None):
        self.support_idx_ = mrmr_rank(X, y, self.k)
        return self
    def transform(self, X):
        return np.asarray(X)[:, self.support_idx_]
# =========================================================================


def make_pipe(cfg, class_weight):
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", MRMRSelector(k=cfg["k_feat"])),
        ("svm",    SVC(kernel="rbf", C=cfg["C"], gamma=cfg["gamma"],
                       class_weight=class_weight, cache_size=1024)),
    ])


def damped_class_weight(y, power=1):
    """Inverse-frequency class weights damped by `power`.
    power=1.0 -> sklearn 'balanced'; power=0.0 -> unweighted; 0.5 -> sqrt.
    Renormalised so the sample-weighted mean weight is 1, so C keeps the
    same effective scale across the ablation. Training labels only."""
    classes, counts = np.unique(y, return_counts=True)
    n = len(y)
    inv = n / (len(classes) * counts)        # 'balanced' inverse-frequency weights
    weights = inv ** power                    # apply the damping exponent
    weights *= n / np.sum(counts * weights)   # renormalise mean sample weight -> 1
    return {int(c): float(w) for c, w in zip(classes, weights)}

## Load one patient

Returns a `seizure_records` dict `{record_name: (X, y)}` for the seizure-containing files only (records with at least one seizure window).

In [3]:
IMAG_TOL = 1e-6   # |imag| above this = invalid GC window, dropped


def _read_feats(h, key):
    """Read a features array, handling MATLAB-complex compound dtype.
    Returns (X_real_oriented_rows=windows, bad_window_mask)."""
    X = h[key][:]
    if X.dtype.names and "imag" in X.dtype.names:
        real, imag = X["real"], X["imag"]
    else:
        real, imag = X, np.zeros_like(X)
    if real.shape[0] == N_FEATURES:          # orient rows = windows
        real, imag = real.T, imag.T
    bad = np.abs(imag).max(axis=1) > IMAG_TOL
    return np.ascontiguousarray(real, dtype=np.float64), bad


def load_patient(patient):
    """Load every seizure-containing record for `patient` as {record: (X, y)}.
    Only files with at least one seizure window are kept."""
    records = {}
    for f in sorted(SEIZURE_DIR.glob(f"{patient}*_seizure.mat")):
        try:
            with h5py.File(f, "r") as h:
                X, bad = _read_feats(h, FEATURE_KEY)
                yv = h[LABEL_KEY][:].squeeze()
        except OSError as e:
            print(f"  [SKIP corrupt] {f.name} — {e}")
            continue
        if yv.shape[0] == X.shape[0]:
            y = yv.astype(int)
        else:
            y = yv[: X.shape[0]].astype(int)
        if bad.any():
            print(f"  [DROP complex] {f.name}: {int(bad.sum())} window(s) "
                  f"{list(np.where(bad)[0])}")
            X, y = X[~bad], y[~bad]
        records[f.stem.replace("_seizure", "")] = (X, y)
    return {k: v for k, v in records.items() if v[1].sum() > 0}

## Event-scoring helpers

In [4]:
def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN


def windows_to_mask(scores, duration, threshold, k):
    pos = (scores >= threshold).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= k:
            alarm[i - k + 1: i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n


def ref_mask_from_record(record, n, offset_sec=0.0):
    # offset_sec shifts absolute annotation times into the local coordinates of
    # a record that starts offset_sec into the file. All current callers use the
    # default 0.0 (full records); the parameter is retained for that call signature.
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round((r["start_sec"] - offset_sec) * GRID_FS))
        e = int(round((r["end_sec"]   - offset_sec) * GRID_FS))
        if e <= 0 or s >= n:
            continue
        mask[max(s, 0):min(e, n)] = True
    return mask


def _arming_times(sc, thr, k):
    pos = (sc >= thr).astype(int)
    arm, run = [], 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:
            arm.append(i * STEP + WINDOW_LEN)
    return arm


def detection_latencies(record, sc, thr, k):
    TOL_PRE, TOL_POST = 30, 60
    arm_times = _arming_times(sc, thr, k)
    lat = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in arm_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        if cands:
            lat.append(min(cands) - onset)
    return lat


def _safe_div(a, b):
    return a / b if b else float("nan")


def segment_metrics(y, sc, thr):
    y = np.asarray(y).astype(int)
    pred = (sc >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    acc  = _safe_div(tp + tn, tp + tn + fp + fn)
    spec = _safe_div(tn, tn + fp)
    sens = _safe_div(tp, tp + fn)
    prec = _safe_div(tp, tp + fp)
    f1 = (_safe_div(2 * prec * sens, prec + sens)
          if (prec == prec and sens == sens) else float("nan"))
    return acc, spec, sens, prec, f1, tp, fp, fn


def event_detections(record, sc, thr, k):
    """One row per ground-truth seizure EVENT in `record`: whether the
    operating point (thr, k) caught it and, if so, the onset latency.
    Uses the SAME arming logic / tolerance window as detection_latencies,
    so the flag and latency stay consistent with the reported metrics."""
    TOL_PRE, TOL_POST = 30, 60
    arm_times = _arming_times(sc, thr, k)
    rows = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in arm_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        rows.append({
            "event_start": onset,
            "event_end":   offset,
            "detected":    1 if cands else 0,
            "latency_s":   (min(cands) - onset) if cands else float("nan"),
        })
    return rows


def event_metrics_at_k(test_outputs, threshold, k):
    """Event-level TP/FP/FN + metrics over ALL test records at (threshold, k).
    Cheap re-scoring of already-computed scores -> a-priori k-robustness table."""
    TP = FP = FN = 0
    for record, sc, y, dur, *_ in test_outputs:
        hyp, n = windows_to_mask(sc, dur, threshold, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS),
                                 Annotation(hyp, GRID_FS), SZCORE_PARAMS)
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    total_days = sum(t[3] for t in test_outputs) / 86400
    sens = TP/(TP+FN) if (TP+FN) else 0.0
    prec = TP/(TP+FP) if (TP+FP) else 0.0
    f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
    fpday = FP/total_days if total_days else float("nan")
    return dict(k=k, TP=TP, FP=FP, FN=FN, sensitivity=sens,
                precision=prec, f1=f1, fp_per_day=fpday)


## LORO over seizure files

For each seizure file: hold it out, fit the fixed pipeline on every window of all the other seizure files (trained in full), then score the held-out file in full at the fixed operating point (threshold 0, k = 3). `roc_auc` / `ap` are per-fold diagnostics only — printed, never used to choose anything.

In [5]:
def build_train(seizure_records, exclude=None):
    """Stack every window of all seizure-containing records EXCEPT `exclude`
    (the held-out LORO test record). No undersampling -- trained in full."""
    Xs, ys = [], []
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        Xs.append(X)
        ys.append(np.asarray(y))
    return np.vstack(Xs), np.concatenate(ys)


def diag_scores(y, sc):
    """Per-fold diagnostics (ROC-AUC, average precision). NaN if the held-out
    record is single-class. Reported only, never used to select anything."""
    y = np.asarray(y).astype(int)
    if y.sum() == 0 or y.sum() == len(y):
        return float("nan"), float("nan")
    return roc_auc_score(y, sc), average_precision_score(y, sc)


def process_seizure_fold(seizure_records, test_record, cfg, k, thr):
    """One LORO fold: fit the fixed pipeline on all OTHER seizure records (in
    full), then score the held-out record in full at the fixed operating point
    (thr, k). No tuning."""
    Xtr, ytr = build_train(seizure_records, exclude=test_record)
    cw = damped_class_weight(ytr, CLASS_WEIGHT_POWER)
    model = make_pipe(cfg, cw).fit(Xtr, ytr)
    Xte, yte = seizure_records[test_record]
    sc = model.decision_function(Xte)
    roc, ap = diag_scores(yte, sc)
    return dict(record=test_record, sc=sc, y=yte, dur=duration_of(yte),
                thr=thr, k=k, roc=roc, ap=ap,
                C=cfg["C"], gamma=cfg["gamma"], k_feat=cfg["k_feat"])

## Run one patient (build tables + return summary row)

In [6]:
def run_patient(patient, cfg):
    seizure_records = load_patient(patient)
    if not seizure_records:
        print(f"[{patient}] no seizure records — skipping")
        return None, None, None, None
    print(f"[{patient}] seizure files = {len(seizure_records)}")

    # LORO over seizure files: hold out one, train on the rest (in full), score
    # the held-out record in full at the fixed operating point (thr=0, k=3).
    results = Parallel(n_jobs=OUTER_N_JOBS)(
        delayed(process_seizure_fold)(
            seizure_records, r, cfg, FIXED_K, FIXED_THRESHOLD)
        for r in seizure_records.keys()
    )

    test_outputs, meta = [], {}
    for res in results:
        test_outputs.append((res["record"], res["sc"], res["y"], res["dur"],
                             res["thr"], res["k"]))
        meta[res["record"]] = dict(roc=res["roc"], ap=res["ap"],
                                   C=res["C"], gamma=res["gamma"],
                                   k_feat=res["k_feat"])
        print(f"  [SEIZ] {res['record']}  thr={res['thr']:.3f} k={res['k']}  "
              f"C={res['C']} g={res['gamma']} p={res['k_feat']}  "
              f"roc={res['roc']:.3f} ap={res['ap']:.3f}  "
              f"test_pos={int(res['y'].sum())}")

    # --- per-record pass: segment table + event totals (all records seizure) ---
    TP = FP = FN = 0
    sTP = sFP = sFN = 0   # 1 Hz sample-based accumulators (SzCORE 2.3)
    all_latencies, segment_rows, event_rows, fp_rows = [], [], [], []
    for record, sc, y, dur, thr, k in test_outputs:
        hyp, n = windows_to_mask(sc, dur, thr, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS),
                                 Annotation(hyp, GRID_FS), SZCORE_PARAMS)
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
        # 1 Hz sample-based scoring on the SAME masks (no tolerances/merge).
        ss = scoring.SampleScoring(Annotation(ref, GRID_FS),
                                   Annotation(hyp, GRID_FS))
        sTP += ss.tp; sFP += ss.fp; sFN += ss.refTrue - ss.tp
        # False-positive table: one row per hypothesis (alarm) event that
        # SzCORE counts as a false detection.
        for ev in s.hyp.events:
            a, b = round(ev[0] * s.fs), round(ev[1] * s.fs)
            if np.all(~s.tpMask[a:b]):
                fp_rows.append({
                    "patient": patient, "record": record,
                    "fp_start": ev[0], "fp_duration_s": ev[1] - ev[0],
                })
        all_latencies.extend(detection_latencies(record, sc, thr, k))
        # Event-level table: one row per ground-truth seizure event.
        for ev in event_detections(record, sc, thr, k):
            event_rows.append({
                "patient": patient, "record": record,
                "event_start": ev["event_start"],
                "event_end":   ev["event_end"],
                "detected":    ev["detected"],
                "latency_s":   ev["latency_s"],
                "threshold": thr, "k": k,
                "k_feat": meta[record]["k_feat"],
            })

        acc, spec, sens, prec, f1, stp, sfp, sfn = segment_metrics(y, sc, 0.0)
        segment_rows.append({
            "C": meta[record]["C"], "gamma": meta[record]["gamma"],
            "k_feat": meta[record]["k_feat"],
            "patient": patient, "record": record,
            "n_windows": int(len(y)), "n_seizure_windows": int(np.sum(y)),
            "record_length": dur, "threshold": thr, "k": k,
            "roc_auc": meta[record]["roc"], "ap": meta[record]["ap"],
            "seg_accuracy": acc, "seg_specificity": spec,
            "seg_sensitivity": sens, "seg_precision": prec, "seg_f1": f1,
            "tp": stp, "fp": sfp, "fn": sfn,
        })
        print(f"    {record} done  (SEIZ)", flush=True)

    # --- event-level totals ---
    sens = TP/(TP+FN) if (TP+FN) else 0.0
    prec = TP/(TP+FP) if (TP+FP) else 0.0
    f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
    # 1 Hz sample-based metrics (SzCORE 2.3; complementary to event-based).
    s_sens = sTP/(sTP+sFN) if (sTP+sFN) else 0.0
    s_prec = sTP/(sTP+sFP) if (sTP+sFP) else 0.0
    s_f1   = 2*s_sens*s_prec/(s_sens+s_prec) if (s_sens+s_prec) else 0.0
    total_len_sec = sum(t[3] for t in test_outputs)
    days = total_len_sec / 86400
    med_lat = float(np.median(all_latencies)) if all_latencies else float("nan")

    fp_day = FP / days if days else float("nan")
    print(f"  EVENT: TP={TP} FP={FP} FN={FN}  sens={sens:.3f} "
          f"prec={prec:.3f} f1={f1:.3f}  FP/day={fp_day:.3f}  "
          f"lat={med_lat:.1f}s")
    print(f"  SAMPLE(1Hz): TP={sTP} FP={sFP} FN={sFN}  "
          f"sens={s_sens:.3f} prec={s_prec:.3f} f1={s_f1:.3f}")

    # --- a-priori k-robustness sweep (threshold fixed at 0; reporting only) ---
    ksweep = [event_metrics_at_k(test_outputs, FIXED_THRESHOLD, kk) for kk in K_SWEEP]
    print("  K-SWEEP (thr=0):")
    for m in ksweep:
        print(f"    k={m['k']}: sens={m['sensitivity']:.3f} "
              f"prec={m['precision']:.3f} f1={m['f1']:.3f} "
              f"FP/day={m['fp_per_day']:.3f}  (TP={m['TP']} FP={m['FP']} FN={m['FN']})")

    summary_row = {
        "C": cfg["C"], "gamma": cfg["gamma"], "k_feat": cfg["k_feat"],
        "patient": patient,
        "total_length_hours": total_len_sec / 3600,
        "TP": TP, "FP": FP, "FN": FN,
        "sensitivity": sens, "precision": prec, "f1": f1,
        "sample_TP": sTP, "sample_FP": sFP, "sample_FN": sFN,
        "sample_sensitivity": s_sens, "sample_precision": s_prec, "sample_f1": s_f1,
        "FP/day": FP / days if days else float("nan"),
        "FP/hour": FP / (days * 24) if days else float("nan"),
        "latency_median": med_lat, "n_detected": len(all_latencies),
        "mean_fold_roc_auc": float(np.nanmean(
            [m["roc"] for m in meta.values()])),
        "mean_fold_ap": float(np.nanmean([m["ap"] for m in meta.values()])),
    }
    for m in ksweep:
        kk = m["k"]
        summary_row[f"sens_k{kk}"] = m["sensitivity"]
        summary_row[f"f1_k{kk}"]   = m["f1"]
        summary_row[f"fpday_k{kk}"] = m["fp_per_day"]

    return summary_row, segment_rows, event_rows, fp_rows

# Run

Fixed operating point (threshold 0, k = 3), no tuning. Seizure-containing files only. Runs every patient in `PATIENTS` (set the `PATIENT_SHARD` env var to subset).

## chb01

In [8]:
PATIENT = "chb01"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb01
[chb01] seizure files = 7
  [SEIZ] chb01_03  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.977 ap=0.394  test_pos=16
  [SEIZ] chb01_04  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.999 ap=0.862  test_pos=11
  [SEIZ] chb01_15  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.759 ap=0.106  test_pos=16
  [SEIZ] chb01_16  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.997 ap=0.765  test_pos=21
  [SEIZ] chb01_18  thr=0.000 k=3  C=1 g=0.001 p=100  roc=1.000 ap=0.989  test_pos=37
  [SEIZ] chb01_21  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.997 ap=0.974  test_pos=38
  [SEIZ] chb01_26  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.991 ap=0.942  test_pos=41
    chb01_03 done  (SEIZ)
    chb01_04 done  (SEIZ)
    chb01_15 done  (SEIZ)
    chb01_16 done  (SEIZ)
    chb01_18 done  (SEIZ)
    chb01_21 done  (SEIZ)
    chb01_26 done  (SEIZ)
  EVENT: TP=7 FP=8 FN=0  sens=1.000 prec=0.467 f1=0.636  FP/day=28.890  lat=14.0s
  SAMPLE(1Hz): TP=384 FP=360 FN=58  sens=0.869 pr

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb01,6.645833,7,8,0,1.0,0.466667,...,0.719034,1.0,0.608696,32.501567,1.0,0.636364,28.890282,1.0,0.736842,18.056426


In [7]:
PATIENT = "chb02"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb02
[chb02] seizure files = 3
  [SEIZ] chb02_16+  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.633 ap=0.034  test_pos=33
  [SEIZ] chb02_16  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.661 ap=0.165  test_pos=33
  [SEIZ] chb02_19  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.872 ap=0.019  test_pos=4
    chb02_16+ done  (SEIZ)
    chb02_16 done  (SEIZ)
    chb02_19 done  (SEIZ)
  EVENT: TP=1 FP=9 FN=2  sens=0.333 prec=0.100 f1=0.154  FP/day=95.323  lat=-21.5s
  SAMPLE(1Hz): TP=0 FP=306 FN=172  sens=0.000 prec=0.000 f1=0.000
  K-SWEEP (thr=0):
    k=2: sens=0.333 prec=0.067 f1=0.111 FP/day=148.281  (TP=1 FP=14 FN=2)
    k=3: sens=0.333 prec=0.100 f1=0.154 FP/day=95.323  (TP=1 FP=9 FN=2)
    k=4: sens=0.333 prec=0.143 f1=0.200 FP/day=63.549  (TP=1 FP=6 FN=2)
  >>> SAVED [chb02] -> RESULTS_SUMMARY_chb02.csv

Wrote RESULTS_SUMMARY_chb02.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision    FP/day
  1   9   2     0.333333        0.1 95.

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb02,2.265972,1,9,2,0.333333,0.1,...,0.072642,0.333333,0.111111,148.280723,0.333333,0.153846,95.323322,0.333333,0.2,63.548881


In [9]:
PATIENT = "chb03"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb03
[chb03] seizure files = 7
  [SEIZ] chb03_01  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.917 ap=0.182  test_pos=21
  [SEIZ] chb03_02  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.975 ap=0.513  test_pos=26
  [SEIZ] chb03_03  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.983 ap=0.699  test_pos=28
  [SEIZ] chb03_04  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.987 ap=0.759  test_pos=21
  [SEIZ] chb03_34  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.927 ap=0.105  test_pos=19
  [SEIZ] chb03_35  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.964 ap=0.319  test_pos=26
  [SEIZ] chb03_36  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.858 ap=0.073  test_pos=22
    chb03_01 done  (SEIZ)
    chb03_02 done  (SEIZ)
    chb03_03 done  (SEIZ)
    chb03_04 done  (SEIZ)
    chb03_34 done  (SEIZ)
    chb03_35 done  (SEIZ)
    chb03_36 done  (SEIZ)
  EVENT: TP=6 FP=11 FN=1  sens=0.857 prec=0.353 f1=0.500  FP/day=37.714  lat=21.8s
  SAMPLE(1Hz): TP=183 FP=203 FN=219  sens=0.455 

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb03,7.0,6,11,1,0.857143,0.352941,...,0.378513,1.0,0.466667,54.857143,0.857143,0.5,37.714286,0.571429,0.533333,13.714286


In [10]:
PATIENT = "chb04"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb04
[chb04] seizure files = 3
  [SEIZ] chb04_05  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.976 ap=0.484  test_pos=20
  [SEIZ] chb04_08  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.895 ap=0.144  test_pos=44
  [SEIZ] chb04_28  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.740 ap=0.032  test_pos=88
    chb04_05 done  (SEIZ)
    chb04_08 done  (SEIZ)
    chb04_28 done  (SEIZ)
  EVENT: TP=3 FP=36 FN=1  sens=0.750 prec=0.077 f1=0.140  FP/day=81.090  lat=19.0s
  SAMPLE(1Hz): TP=125 FP=812 FN=253  sens=0.331 prec=0.133 f1=0.190
  K-SWEEP (thr=0):
    k=2: sens=0.750 prec=0.050 f1=0.094 FP/day=128.392  (TP=3 FP=57 FN=1)
    k=3: sens=0.750 prec=0.077 f1=0.140 FP/day=81.090  (TP=3 FP=36 FN=1)
    k=4: sens=0.500 prec=0.091 f1=0.154 FP/day=45.050  (TP=2 FP=20 FN=2)
  >>> SAVED [chb04] -> RESULTS_SUMMARY_chb04.csv

Wrote RESULTS_SUMMARY_chb04.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision    FP/day
  3  36   1         0.75   0.076923 

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb04,10.654861,3,36,1,0.75,0.076923,...,0.220085,0.75,0.09375,128.392101,0.75,0.139535,81.089748,0.5,0.153846,45.04986


In [11]:
PATIENT = "chb05"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb05
[chb05] seizure files = 5
  [SEIZ] chb05_06  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.988 ap=0.827  test_pos=46
  [SEIZ] chb05_13  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.993 ap=0.829  test_pos=44
  [SEIZ] chb05_16  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.954 ap=0.728  test_pos=39
  [SEIZ] chb05_17  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.921 ap=0.450  test_pos=48
  [SEIZ] chb05_22  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.956 ap=0.708  test_pos=47
    chb05_06 done  (SEIZ)
    chb05_13 done  (SEIZ)
    chb05_16 done  (SEIZ)
    chb05_17 done  (SEIZ)
    chb05_22 done  (SEIZ)
  EVENT: TP=5 FP=3 FN=0  sens=1.000 prec=0.625 f1=0.769  FP/day=14.400  lat=16.5s
  SAMPLE(1Hz): TP=426 FP=93 FN=132  sens=0.763 prec=0.821 f1=0.791
  K-SWEEP (thr=0):
    k=2: sens=1.000 prec=0.312 f1=0.476 FP/day=52.800  (TP=5 FP=11 FN=0)
    k=3: sens=1.000 prec=0.625 f1=0.769 FP/day=14.400  (TP=5 FP=3 FN=0)
    k=4: sens=1.000 prec=0.714 f1=0.833 

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb05,5.0,5,3,0,1.0,0.625,...,0.708275,1.0,0.47619,52.8,1.0,0.769231,14.4,1.0,0.833333,9.6


In [12]:
PATIENT = "chb06"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb06
[chb06] seizure files = 7
  [SEIZ] chb06_01  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.892 ap=0.311  test_pos=19
  [SEIZ] chb06_04  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.755 ap=0.014  test_pos=16
  [SEIZ] chb06_09  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.967 ap=0.091  test_pos=7
  [SEIZ] chb06_10  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.978 ap=0.352  test_pos=5
  [SEIZ] chb06_13  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.992 ap=0.154  test_pos=5
  [SEIZ] chb06_18  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.425 ap=0.002  test_pos=5
  [SEIZ] chb06_24  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.801 ap=0.006  test_pos=7
    chb06_01 done  (SEIZ)
    chb06_04 done  (SEIZ)
    chb06_09 done  (SEIZ)
    chb06_10 done  (SEIZ)
    chb06_13 done  (SEIZ)
    chb06_18 done  (SEIZ)
    chb06_24 done  (SEIZ)
  EVENT: TP=1 FP=8 FN=9  sens=0.100 prec=0.111 f1=0.105  FP/day=7.415  lat=66.0s
  SAMPLE(1Hz): TP=0 FP=139 FN=153  sens=0.000 prec=0.00

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb06,25.892361,1,8,9,0.1,0.111111,...,0.132978,0.4,0.242424,17.611372,0.1,0.105263,7.415314,0.1,0.181818,0.0


In [13]:
PATIENT = "chb07"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb07
[chb07] seizure files = 3
  [SEIZ] chb07_12  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.969 ap=0.334  test_pos=35
  [SEIZ] chb07_13  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.882 ap=0.595  test_pos=39
  [SEIZ] chb07_19  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.846 ap=0.099  test_pos=76
    chb07_12 done  (SEIZ)
    chb07_13 done  (SEIZ)
    chb07_19 done  (SEIZ)
  EVENT: TP=2 FP=23 FN=1  sens=0.667 prec=0.080 f1=0.143  FP/day=53.255  lat=13.8s
  SAMPLE(1Hz): TP=112 FP=455 FN=213  sens=0.345 prec=0.198 f1=0.251
  K-SWEEP (thr=0):
    k=2: sens=0.667 prec=0.054 f1=0.100 FP/day=81.040  (TP=2 FP=35 FN=1)
    k=3: sens=0.667 prec=0.080 f1=0.143 FP/day=53.255  (TP=2 FP=23 FN=1)
    k=4: sens=0.667 prec=0.167 f1=0.267 FP/day=23.154  (TP=2 FP=10 FN=1)
  >>> SAVED [chb07] -> RESULTS_SUMMARY_chb07.csv

Wrote RESULTS_SUMMARY_chb07.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision    FP/day
  2  23   1     0.666667       0.08 5

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb07,10.365278,2,23,1,0.666667,0.08,...,0.342386,0.666667,0.1,81.039796,0.666667,0.142857,53.254723,0.666667,0.266667,23.154228


In [14]:
PATIENT = "chb08"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb08
[chb08] seizure files = 5
  [SEIZ] chb08_02  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.974 ap=0.704  test_pos=69
  [SEIZ] chb08_05  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.945 ap=0.587  test_pos=76
  [SEIZ] chb08_11  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.971 ap=0.760  test_pos=53
  [SEIZ] chb08_13  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.965 ap=0.660  test_pos=64
  [SEIZ] chb08_21  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.866 ap=0.535  test_pos=105
    chb08_02 done  (SEIZ)
    chb08_05 done  (SEIZ)
    chb08_11 done  (SEIZ)
    chb08_13 done  (SEIZ)
    chb08_21 done  (SEIZ)
  EVENT: TP=5 FP=13 FN=0  sens=1.000 prec=0.278 f1=0.435  FP/day=62.400  lat=22.0s
  SAMPLE(1Hz): TP=664 FP=518 FN=255  sens=0.723 prec=0.562 f1=0.632
  K-SWEEP (thr=0):
    k=2: sens=1.000 prec=0.227 f1=0.370 FP/day=81.600  (TP=5 FP=17 FN=0)
    k=3: sens=1.000 prec=0.278 f1=0.435 FP/day=62.400  (TP=5 FP=13 FN=0)
    k=4: sens=1.000 prec=0.333 f1=0.

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb08,5.0,5,13,0,1.0,0.277778,...,0.649035,1.0,0.37037,81.6,1.0,0.434783,62.4,1.0,0.5,48.0


In [15]:
PATIENT = "chb09"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb09
[chb09] seizure files = 3
  [SEIZ] chb09_06  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.925 ap=0.694  test_pos=26
  [SEIZ] chb09_08  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.986 ap=0.805  test_pos=60
  [SEIZ] chb09_19  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.974 ap=0.433  test_pos=25
    chb09_06 done  (SEIZ)
    chb09_08 done  (SEIZ)
    chb09_19 done  (SEIZ)
  EVENT: TP=4 FP=6 FN=0  sens=1.000 prec=0.400 f1=0.571  FP/day=15.027  lat=21.5s
  SAMPLE(1Hz): TP=215 FP=186 FN=61  sens=0.779 prec=0.536 f1=0.635
  K-SWEEP (thr=0):
    k=2: sens=1.000 prec=0.235 f1=0.381 FP/day=32.559  (TP=4 FP=13 FN=0)
    k=3: sens=1.000 prec=0.400 f1=0.571 FP/day=15.027  (TP=4 FP=6 FN=0)
    k=4: sens=1.000 prec=0.571 f1=0.727 FP/day=7.514  (TP=4 FP=3 FN=0)
  >>> SAVED [chb09] -> RESULTS_SUMMARY_chb09.csv

Wrote RESULTS_SUMMARY_chb09.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision    FP/day
  4   6   0          1.0        0.4 15.027

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb09,9.582639,4,6,0,1.0,0.4,...,0.643902,1.0,0.380952,32.558881,1.0,0.571429,15.027176,1.0,0.727273,7.513588


In [16]:
PATIENT = "chb10"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb10
[chb10] seizure files = 7
  [SEIZ] chb10_12  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.975 ap=0.187  test_pos=14
  [SEIZ] chb10_20  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.918 ap=0.501  test_pos=28
  [SEIZ] chb10_27  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.990 ap=0.679  test_pos=26
  [SEIZ] chb10_30  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.963 ap=0.382  test_pos=23
  [SEIZ] chb10_31  thr=0.000 k=3  C=1 g=0.001 p=100  roc=1.000 ap=0.969  test_pos=30
  [SEIZ] chb10_38  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.978 ap=0.618  test_pos=35
  [SEIZ] chb10_89  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.867 ap=0.246  test_pos=21
    chb10_12 done  (SEIZ)
    chb10_20 done  (SEIZ)
    chb10_27 done  (SEIZ)
    chb10_30 done  (SEIZ)
    chb10_31 done  (SEIZ)
    chb10_38 done  (SEIZ)
    chb10_89 done  (SEIZ)
  EVENT: TP=6 FP=13 FN=1  sens=0.857 prec=0.316 f1=0.462  FP/day=22.259  lat=13.5s
  SAMPLE(1Hz): TP=282 FP=261 FN=165  sens=0.631 

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb10,14.016667,6,13,1,0.857143,0.315789,...,0.511509,0.857143,0.307692,44.51843,0.857143,0.461538,22.259215,0.714286,0.555556,10.273484


In [17]:
PATIENT = "chb11"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb11
[chb11] seizure files = 3
  [SEIZ] chb11_82  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.998 ap=0.710  test_pos=9
  [SEIZ] chb11_92  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.871 ap=0.087  test_pos=13
  [SEIZ] chb11_99  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.887 ap=0.779  test_pos=301
    chb11_82 done  (SEIZ)
    chb11_92 done  (SEIZ)
    chb11_99 done  (SEIZ)
  EVENT: TP=5 FP=3 FN=0  sens=1.000 prec=0.625 f1=0.769  FP/day=25.772  lat=1.0s
  SAMPLE(1Hz): TP=180 FP=102 FN=626  sens=0.223 prec=0.638 f1=0.331
  K-SWEEP (thr=0):
    k=2: sens=1.000 prec=0.455 f1=0.625 FP/day=51.544  (TP=5 FP=6 FN=0)
    k=3: sens=1.000 prec=0.625 f1=0.769 FP/day=25.772  (TP=5 FP=3 FN=0)
    k=4: sens=0.800 prec=0.667 f1=0.727 FP/day=17.181  (TP=4 FP=2 FN=1)
  >>> SAVED [chb11] -> RESULTS_SUMMARY_chb11.csv

Wrote RESULTS_SUMMARY_chb11.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision    FP/day
  5   3   0          1.0      0.625 25.771

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb11,2.79375,5,3,0,1.0,0.625,...,0.524915,1.0,0.625,51.543624,1.0,0.769231,25.771812,0.8,0.727273,17.181208


In [18]:
PATIENT = "chb12"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb12
[chb12] seizure files = 10
  [SEIZ] chb12_06  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.679 ap=0.187  test_pos=38
  [SEIZ] chb12_08  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.841 ap=0.528  test_pos=32
  [SEIZ] chb12_09  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.973 ap=0.503  test_pos=26
  [SEIZ] chb12_10  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.962 ap=0.748  test_pos=31
  [SEIZ] chb12_11  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.833 ap=0.084  test_pos=15
  [SEIZ] chb12_23  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.635 ap=0.157  test_pos=88
  [SEIZ] chb12_33  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.855 ap=0.196  test_pos=19
  [SEIZ] chb12_36  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.858 ap=0.124  test_pos=11
  [SEIZ] chb12_38  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.826 ap=0.468  test_pos=77
  [SEIZ] chb12_42  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.909 ap=0.513  test_pos=64
    chb12_06 done  (SEIZ)
    chb12_08 done  (SEIZ)
    c

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb12,9.681944,20,14,7,0.740741,0.588235,...,0.350828,0.925926,0.694444,49.576818,0.740741,0.655738,34.703773,0.666667,0.642857,27.26725


In [19]:
PATIENT = "chb13"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb13
[chb13] seizure files = 8
  [SEIZ] chb13_19  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.606 ap=0.034  test_pos=18
  [SEIZ] chb13_21  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.789 ap=0.091  test_pos=28
  [SEIZ] chb13_40  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.483 ap=0.026  test_pos=39
  [SEIZ] chb13_55  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.591 ap=0.017  test_pos=15
  [SEIZ] chb13_58  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.846 ap=0.017  test_pos=7
  [SEIZ] chb13_59  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.678 ap=0.039  test_pos=25
  [SEIZ] chb13_60  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.902 ap=0.105  test_pos=9
  [SEIZ] chb13_62  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.479 ap=0.054  test_pos=75
    chb13_19 done  (SEIZ)
    chb13_21 done  (SEIZ)
    chb13_40 done  (SEIZ)
    chb13_55 done  (SEIZ)
    chb13_58 done  (SEIZ)
    chb13_59 done  (SEIZ)
    chb13_60 done  (SEIZ)
    chb13_62 done  (SEIZ)
  EVENT: TP=0 FP=1 FN=

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb13,8.0,0,1,12,0.0,0.0,...,0.047842,0.083333,0.111111,15.0,0.0,0.0,3.0,0.0,0.0,0.0


In [20]:
PATIENT = "chb14"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb14
[chb14] seizure files = 7
  [SEIZ] chb14_03  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.972 ap=0.222  test_pos=6
  [SEIZ] chb14_04  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.967 ap=0.236  test_pos=17
  [SEIZ] chb14_06  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.899 ap=0.093  test_pos=6
  [SEIZ] chb14_11  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.861 ap=0.293  test_pos=16
  [SEIZ] chb14_17  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.948 ap=0.139  test_pos=8
  [SEIZ] chb14_18  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.982 ap=0.550  test_pos=9
  [SEIZ] chb14_27  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.983 ap=0.294  test_pos=6
    chb14_03 done  (SEIZ)
    chb14_04 done  (SEIZ)
    chb14_06 done  (SEIZ)
    chb14_11 done  (SEIZ)
    chb14_17 done  (SEIZ)
    chb14_18 done  (SEIZ)
    chb14_27 done  (SEIZ)
  EVENT: TP=2 FP=6 FN=6  sens=0.250 prec=0.250 f1=0.250  FP/day=20.571  lat=12.0s
  SAMPLE(1Hz): TP=29 FP=120 FN=140  sens=0.172 prec=0.

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb14,7.0,2,6,6,0.25,0.25,...,0.26106,0.625,0.416667,37.714286,0.25,0.25,20.571429,0.25,0.285714,13.714286


In [21]:
PATIENT = "chb15"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb15
[chb15] seizure files = 14
  [SEIZ] chb15_06  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.989 ap=0.715  test_pos=50
  [SEIZ] chb15_10  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.978 ap=0.190  test_pos=13
  [SEIZ] chb15_15  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.844 ap=0.303  test_pos=63
  [SEIZ] chb15_17  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.779 ap=0.026  test_pos=15
  [SEIZ] chb15_20  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.817 ap=0.040  test_pos=22
  [SEIZ] chb15_22  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.773 ap=0.167  test_pos=83
  [SEIZ] chb15_28  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.930 ap=0.571  test_pos=76
  [SEIZ] chb15_31  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.971 ap=0.786  test_pos=48
  [SEIZ] chb15_40  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.905 ap=0.619  test_pos=97
  [SEIZ] chb15_46  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.971 ap=0.830  test_pos=43
  [SEIZ] chb15_49  thr=0.000 k=3  C=1 g=0.001 p=100  roc=

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb15,14.009028,15,34,5,0.75,0.306122,...,0.460733,0.8,0.323232,107.930402,0.75,0.434783,58.248153,0.75,0.545455,34.26362


In [22]:
PATIENT = "chb16"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb16
[chb16] seizure files = 6
  [SEIZ] chb16_10  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.775 ap=0.009  test_pos=4
  [SEIZ] chb16_11  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.930 ap=0.066  test_pos=4
  [SEIZ] chb16_14  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.201 ap=0.003  test_pos=6
  [SEIZ] chb16_16  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.891 ap=0.012  test_pos=3
  [SEIZ] chb16_17  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.673 ap=0.061  test_pos=15
  [SEIZ] chb16_18  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.911 ap=0.043  test_pos=7
    chb16_10 done  (SEIZ)
    chb16_11 done  (SEIZ)
    chb16_14 done  (SEIZ)
    chb16_16 done  (SEIZ)
    chb16_17 done  (SEIZ)
    chb16_18 done  (SEIZ)
  EVENT: TP=0 FP=8 FN=10  sens=0.000 prec=0.000 f1=0.000  FP/day=32.000  lat=nans
  SAMPLE(1Hz): TP=0 FP=146 FN=84  sens=0.000 prec=0.000 f1=0.000
  K-SWEEP (thr=0):
    k=2: sens=0.000 prec=0.000 f1=0.000 FP/day=40.000  (TP=0 FP=10 FN=10)
    k=3

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb16,6.0,0,8,10,0.0,0.0,...,0.032554,0.0,0.0,40.0,0.0,0.0,32.0,0.0,0.0,16.0


In [23]:
PATIENT = "chb17"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb17
[chb17] seizure files = 3
  [SEIZ] chb17a_03  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.878 ap=0.246  test_pos=36
  [SEIZ] chb17a_04  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.790 ap=0.138  test_pos=47
  [SEIZ] chb17b_63  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.972 ap=0.639  test_pos=35
    chb17a_03 done  (SEIZ)
    chb17a_04 done  (SEIZ)
    chb17b_63 done  (SEIZ)
  EVENT: TP=2 FP=8 FN=1  sens=0.667 prec=0.200 f1=0.308  FP/day=63.867  lat=32.2s
  SAMPLE(1Hz): TP=105 FP=223 FN=188  sens=0.358 prec=0.320 f1=0.338
  K-SWEEP (thr=0):
    k=2: sens=1.000 prec=0.200 f1=0.333 FP/day=95.800  (TP=3 FP=12 FN=0)
    k=3: sens=0.667 prec=0.200 f1=0.308 FP/day=63.867  (TP=2 FP=8 FN=1)
    k=4: sens=0.667 prec=0.333 f1=0.444 FP/day=31.933  (TP=2 FP=4 FN=1)
  >>> SAVED [chb17] -> RESULTS_SUMMARY_chb17.csv

Wrote RESULTS_SUMMARY_chb17.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision    FP/day
  2   8   1     0.666667        0.

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb17,3.00625,2,8,1,0.666667,0.2,...,0.341001,1.0,0.333333,95.800416,0.666667,0.307692,63.866944,0.666667,0.444444,31.933472


In [24]:
PATIENT = "chb18"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb18
[chb18] seizure files = 6
  [SEIZ] chb18_29  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.848 ap=0.484  test_pos=20
  [SEIZ] chb18_30  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.530 ap=0.009  test_pos=12
  [SEIZ] chb18_31  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.940 ap=0.695  test_pos=28
  [SEIZ] chb18_32  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.991 ap=0.591  test_pos=22
  [SEIZ] chb18_35  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.811 ap=0.175  test_pos=27
  [SEIZ] chb18_36  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.914 ap=0.518  test_pos=18
    chb18_29 done  (SEIZ)
    chb18_30 done  (SEIZ)
    chb18_31 done  (SEIZ)
    chb18_32 done  (SEIZ)
    chb18_35 done  (SEIZ)
    chb18_36 done  (SEIZ)
  EVENT: TP=4 FP=8 FN=2  sens=0.667 prec=0.333 f1=0.444  FP/day=34.095  lat=20.0s
  SAMPLE(1Hz): TP=159 FP=222 FN=158  sens=0.502 prec=0.417 f1=0.456
  K-SWEEP (thr=0):
    k=2: sens=0.667 prec=0.286 f1=0.400 FP/day=42.619  (TP=4 FP=10 FN=2)


,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb18,5.63125,4,8,2,0.666667,0.333333,...,0.412013,0.666667,0.4,42.619312,0.666667,0.444444,34.09545,0.666667,0.533333,21.309656


In [25]:
PATIENT = "chb19"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb19
[chb19] seizure files = 3
  [SEIZ] chb19_28  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.799 ap=0.105  test_pos=31
  [SEIZ] chb19_29  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.933 ap=0.479  test_pos=31
  [SEIZ] chb19_30  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.915 ap=0.315  test_pos=33
    chb19_28 done  (SEIZ)
    chb19_29 done  (SEIZ)
    chb19_30 done  (SEIZ)
  EVENT: TP=3 FP=14 FN=0  sens=1.000 prec=0.176 f1=0.300  FP/day=114.736  lat=18.5s
  SAMPLE(1Hz): TP=153 FP=572 FN=83  sens=0.648 prec=0.211 f1=0.318
  K-SWEEP (thr=0):
    k=2: sens=1.000 prec=0.150 f1=0.261 FP/day=139.322  (TP=3 FP=17 FN=0)
    k=3: sens=1.000 prec=0.176 f1=0.300 FP/day=114.736  (TP=3 FP=14 FN=0)
    k=4: sens=1.000 prec=0.231 f1=0.375 FP/day=81.954  (TP=3 FP=10 FN=0)
  >>> SAVED [chb19] -> RESULTS_SUMMARY_chb19.csv

Wrote RESULTS_SUMMARY_chb19.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision     FP/day
  3  14   0          1.0   0.17647

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb19,2.928472,3,14,0,1.0,0.176471,...,0.299683,1.0,0.26087,139.321793,1.0,0.3,114.735594,1.0,0.375,81.953996


In [26]:
PATIENT = "chb20"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb20
[chb20] seizure files = 6
  [SEIZ] chb20_12  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.990 ap=0.489  test_pos=12
  [SEIZ] chb20_13  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.989 ap=0.702  test_pos=28
  [SEIZ] chb20_14  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.987 ap=0.449  test_pos=15
  [SEIZ] chb20_15  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.980 ap=0.768  test_pos=35
  [SEIZ] chb20_16  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.934 ap=0.154  test_pos=14
  [SEIZ] chb20_68  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.959 ap=0.181  test_pos=15
    chb20_12 done  (SEIZ)
    chb20_13 done  (SEIZ)
    chb20_14 done  (SEIZ)
    chb20_15 done  (SEIZ)
    chb20_16 done  (SEIZ)
    chb20_68 done  (SEIZ)
  EVENT: TP=8 FP=7 FN=0  sens=1.000 prec=0.533 f1=0.696  FP/day=30.191  lat=18.0s
  SAMPLE(1Hz): TP=188 FP=480 FN=106  sens=0.639 prec=0.281 f1=0.391
  K-SWEEP (thr=0):
    k=2: sens=1.000 prec=0.533 f1=0.696 FP/day=30.191  (TP=8 FP=7 FN=0)
 

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb20,5.564583,8,7,0,1.0,0.533333,...,0.457201,1.0,0.695652,30.19094,1.0,0.695652,30.19094,0.875,0.636364,30.19094


In [27]:
PATIENT = "chb21"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb21
[chb21] seizure files = 4
  [SEIZ] chb21_19  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.980 ap=0.881  test_pos=22
  [SEIZ] chb21_20  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.959 ap=0.614  test_pos=20
  [SEIZ] chb21_21  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.982 ap=0.760  test_pos=32
  [SEIZ] chb21_22  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.997 ap=0.625  test_pos=5
    chb21_19 done  (SEIZ)
    chb21_20 done  (SEIZ)
    chb21_21 done  (SEIZ)
    chb21_22 done  (SEIZ)
  EVENT: TP=3 FP=0 FN=1  sens=0.750 prec=1.000 f1=0.857  FP/day=0.000  lat=7.0s
  SAMPLE(1Hz): TP=131 FP=29 FN=68  sens=0.658 prec=0.819 f1=0.730
  K-SWEEP (thr=0):
    k=2: sens=1.000 prec=1.000 f1=1.000 FP/day=0.000  (TP=4 FP=0 FN=0)
    k=3: sens=0.750 prec=1.000 f1=0.857 FP/day=0.000  (TP=3 FP=0 FN=1)
    k=4: sens=0.750 prec=1.000 f1=0.857 FP/day=0.000  (TP=3 FP=0 FN=1)
  >>> SAVED [chb21] -> RESULTS_SUMMARY_chb21.csv

Wrote RESULTS_SUMMARY_chb21.csv

Pool

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb21,3.830556,3,0,1,0.75,1.0,...,0.720191,1.0,1.0,0.0,0.75,0.857143,0.0,0.75,0.857143,0.0


In [28]:
PATIENT = "chb22"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb22
[chb22] seizure files = 3
  [SEIZ] chb22_20  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.994 ap=0.845  test_pos=24
  [SEIZ] chb22_25  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.956 ap=0.709  test_pos=30
  [SEIZ] chb22_38  thr=0.000 k=3  C=1 g=0.001 p=100  roc=1.000 ap=0.992  test_pos=29
    chb22_20 done  (SEIZ)
    chb22_25 done  (SEIZ)
    chb22_38 done  (SEIZ)
  EVENT: TP=3 FP=0 FN=0  sens=1.000 prec=1.000 f1=1.000  FP/day=0.000  lat=17.0s
  SAMPLE(1Hz): TP=169 FP=14 FN=35  sens=0.828 prec=0.923 f1=0.873
  K-SWEEP (thr=0):
    k=2: sens=1.000 prec=0.750 f1=0.857 FP/day=8.000  (TP=3 FP=1 FN=0)
    k=3: sens=1.000 prec=1.000 f1=1.000 FP/day=0.000  (TP=3 FP=0 FN=0)
    k=4: sens=1.000 prec=1.000 f1=1.000 FP/day=0.000  (TP=3 FP=0 FN=0)
  >>> SAVED [chb22] -> RESULTS_SUMMARY_chb22.csv

Wrote RESULTS_SUMMARY_chb22.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision  FP/day
  3   0   0          1.0        1.0     0.0


,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb22,3.0,3,0,0,1.0,1.0,...,0.848546,1.0,0.857143,8.0,1.0,1.0,0.0,1.0,1.0,0.0


In [29]:
PATIENT = "chb23"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb23
[chb23] seizure files = 3
  [SEIZ] chb23_06  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.995 ap=0.869  test_pos=46
  [SEIZ] chb23_08  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.995 ap=0.671  test_pos=28
  [SEIZ] chb23_09  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.965 ap=0.573  test_pos=99
    chb23_06 done  (SEIZ)
    chb23_08 done  (SEIZ)
    chb23_09 done  (SEIZ)
  EVENT: TP=6 FP=3 FN=1  sens=0.857 prec=0.667 f1=0.750  FP/day=8.037  lat=17.0s
  SAMPLE(1Hz): TP=274 FP=144 FN=150  sens=0.646 prec=0.656 f1=0.651
  K-SWEEP (thr=0):
    k=2: sens=1.000 prec=0.438 f1=0.609 FP/day=24.112  (TP=7 FP=9 FN=0)
    k=3: sens=0.857 prec=0.667 f1=0.750 FP/day=8.037  (TP=6 FP=3 FN=1)
    k=4: sens=0.714 prec=0.714 f1=0.714 FP/day=5.358  (TP=5 FP=2 FN=2)
  >>> SAVED [chb23] -> RESULTS_SUMMARY_chb23.csv

Wrote RESULTS_SUMMARY_chb23.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision   FP/day
  6   3   1     0.857143   0.666667 8.037209


,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb23,8.958333,6,3,1,0.857143,0.666667,...,0.704529,1.0,0.608696,24.111628,0.857143,0.75,8.037209,0.714286,0.714286,5.35814


In [30]:
PATIENT = "chb24"          # <-- the single patient to run

print(f"Fixed operating point (thr=0, k=3), no tuning  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


Fixed operating point (thr=0, k=3), no tuning  |  patient = chb24
[chb24] seizure files = 12
  [SEIZ] chb24_01  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.990 ap=0.306  test_pos=11
  [SEIZ] chb24_03  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.992 ap=0.630  test_pos=12
  [SEIZ] chb24_04  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.917 ap=0.123  test_pos=13
  [SEIZ] chb24_06  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.974 ap=0.804  test_pos=10
  [SEIZ] chb24_07  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.838 ap=0.241  test_pos=9
  [SEIZ] chb24_09  thr=0.000 k=3  C=1 g=0.001 p=100  roc=1.000 ap=1.000  test_pos=8
  [SEIZ] chb24_11  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.475 ap=0.158  test_pos=28
  [SEIZ] chb24_13  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.999 ap=0.849  test_pos=6
  [SEIZ] chb24_14  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.900 ap=0.263  test_pos=11
  [SEIZ] chb24_15  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.997 ap=0.656  test_pos=7
  [SEIZ] chb24_17  thr=0.000 k=3  C=1 g=0.001 p=100  roc=0.96

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,mean_fold_ap,sens_k2,f1_k2,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4
0,1,0.001,100,chb24,12.0,9,14,3,0.75,0.391304,...,0.504236,0.75,0.486486,32.0,0.75,0.514286,28.0,0.75,0.580645,20.0
